# Inheco ODTC

The Inheco ODTC (On Deck Thermal Cycler) is a thermocycler designed for automated PCR workflows. It features:

- Precise temperature control (4 – 99 °C, up to 4.4 °C/s ramp rate for the 96-well variant)
- Heated lid to prevent condensation
- Motorized door for automated plate handling
- 96-well and 384-well plate formats

The ODTC communicates over Ethernet using the SiLA 1 (SOAP) protocol. You will need the IP address of the device and network connectivity between your computer and the ODTC.

See the [Inheco ODTC product page](https://www.inheco.com/odtc.html) for hardware details.

## Architecture overview

The ODTC integrates into PLR's v1b1 Capability architecture:

- **`ODTC`** — the top-level `Device` (also a `Resource` with physical dimensions). Owns two capabilities.
- **`odtc.tc`** — the `Thermocycler` capability. All temperature control and protocol execution.
- **`odtc.door`** — the `LoadingTray` capability. Motorized door open/close and plate-access resource.
- **`ODTCThermocyclerBackend.RunProtocolParams`** — device-specific parameters (variant, fluid quantity, PID, etc.) passed as `backend_params` to `run_protocol()`.
- **`Protocol` / `Stage` / `Step` / `Ramp`** — the abstract protocol model. Device-agnostic, composable, fully serializable.

## Setup

In [1]:
from pylabrobot.inheco.odtc import ODTC, DoorStateUnknownError, FluidQuantity
from pylabrobot.inheco.odtc.backend import ODTCThermocyclerBackend
from pylabrobot.inheco.odtc.model import ODTCProtocol
from pylabrobot.capabilities.thermocycling import (
    Protocol,
    Stage,
    Step,
    Ramp,
    Overshoot,
    FULL_SPEED,
)

odtc = ODTC(
    odtc_ip="192.168.1.50",  # replace with your ODTC's IP address
    variant=96,                 # 96 or 384
    name="odtc",
)

await odtc.setup()  # connects, resets, and initialises the device

2026-05-03 21:56:25,826 - pylabrobot.inheco.odtc.odtc - INFO - GetStatus returned state: 'standby'
2026-05-03 21:56:25,826 - pylabrobot.inheco.odtc.odtc - INFO - Device is in standby, calling Initialize...
2026-05-03 21:56:34,275 - pylabrobot.inheco.odtc.odtc - INFO - Device initialized and idle


> **Tip:** Thermocycler operations go through `odtc.tc`; door operations go through `odtc.door`. The `ODTC` device handles the SiLA connection lifecycle.

## Door control

`odtc.door` is a `LoadingTray` capability — it controls the motorized door **and** acts as the plate-holding resource in the PLR resource tree (arms can pick up and place plates via `odtc.door`).

### Door state

The ODTC firmware provides no door-state query command, so PLR tracks state locally:

- **Unknown** (initial, or after reconnect) — `odtc.door.backend.is_open` raises `DoorStateUnknownError`
- **Open** — after a successful `odtc.door.open()`
- **Closed** — after a successful `odtc.door.close()`

State resets to unknown whenever the device connection is re-established, because the physical door position may have changed while disconnected.

In [2]:
await odtc.door.open()

In [5]:
# Check door state (raises DoorStateUnknownError if neither open() nor close() has been called)
try:
    print("Door is open:", odtc.door.backend.is_open)
except DoorStateUnknownError as e:
    print("State unknown:", e)

Door is open: False


In [4]:
await odtc.door.close()

## Temperature sensing

The ODTC has multiple internal sensors. `request_temperatures()` returns all of them at once:

In [6]:
sensors = await odtc.tc.backend.request_temperatures()
print(sensors)

AttributeError: 'ODTCThermocyclerBackend' object has no attribute 'request_temperatures'

The standard `request_block_temperature()` and `request_lid_temperature()` methods are also available:

In [7]:
block_temp = await odtc.tc.request_block_temperature()
lid_temp   = await odtc.tc.request_lid_temperature()
print(f"Block: {block_temp:.1f} °C   Lid: {lid_temp:.1f} °C")

Block: 22.2 °C   Lid: 22.8 °C


## Holding at a constant temperature

`set_block_temperature()` creates an ODTC *pre-method* internally, which ramps the block and lid to the target and holds there indefinitely. The call returns immediately (fire-and-forget). Call `deactivate_block()` to stop.

> **Note:** The first call takes several minutes because the device stabilises the block temperature evenly before entering the steady-state hold.

In [8]:
await odtc.tc.set_block_temperature(37.0)

SiLAError: Command SetParameters failed with code 2003: 'MethodSet error 262: VALIDATION_FAILED'

In [9]:
temp = await odtc.tc.request_block_temperature()
print(f"Block: {temp:.1f} °C")

Block: 22.2 °C


In [10]:
await odtc.tc.deactivate_block()

## Running a PCR protocol

Protocols are built from `Protocol` → `Stage` → `Step` objects. This is the same model across all PLR thermocyclers.

### The protocol model

| Type | Key fields |
|------|------------|
| `Step` | `temperature` (°C), `hold_seconds`, `ramp` (optional `Ramp`) |
| `Ramp` | `rate` (°C/s, default = full device speed), `overshoot` (optional) |
| `Stage` | `steps`, `repeats` (cycling count), `inner_stages` (nested loops) |
| `Protocol` | `stages`, `name`, `lid_temperature` (default lid temp) |

### Defining a protocol

In [11]:
pcr_protocol = Protocol(
    name="StandardPCR",
    lid_temperature=105.0,   # default lid temp for all steps
    stages=[
        # Initial denaturation — 5 min at 95 °C
        Stage(
            steps=[Step(temperature=95.0, hold_seconds=300)],
            repeats=1,
        ),
        # PCR cycling — 30 cycles
        Stage(
            steps=[
                Step(temperature=95.0, hold_seconds=30),   # Denature
                Step(temperature=55.0, hold_seconds=30),   # Anneal
                Step(temperature=72.0, hold_seconds=60),   # Extend
            ],
            repeats=30,
        ),
        # Final extension — 10 min at 72 °C
        Stage(
            steps=[Step(temperature=72.0, hold_seconds=600)],
            repeats=1,
        ),
        # Hold at 4 °C — post_heating=True (default) keeps the block here
        # after the method completes. No explicit infinite-hold step needed.
        Stage(
            steps=[Step(temperature=4.0, hold_seconds=30)],
            repeats=1,
        ),
    ],
)

### ODTC-specific run parameters

`ODTCThermocyclerBackend.RunProtocolParams` carries device config:

| Parameter | Meaning | Default |
|-----------|---------|--------|
| `variant` | 96 or 384 | 96 |
| `fluid_quantity` | `FluidQuantity.UL_10_TO_29` / `UL_30_TO_74` / `UL_75_TO_100` | `UL_30_TO_74` |
| `post_heating` | Keep block warm after method | True |
| `dynamic_pre_method_duration` | Device reports live pre-heat time | True |
| `apply_overshoot` | Auto-compute overshoot for steps without explicit `Ramp.overshoot` | True |
| `name` | Protocol name stored on device (None = scratch) | None |

In [12]:
params = ODTCThermocyclerBackend.RunProtocolParams(
    variant=96,
    fluid_quantity=FluidQuantity.UL_30_TO_74,  # 30–74 µL samples
    post_heating=True,
)

### Running the protocol

In [13]:
await odtc.tc.run_protocol(pcr_protocol, backend_params=params)

OverflowError: cannot convert float infinity to integer

`run_protocol()` is fire-and-forget by default — it uploads the method, starts execution, and returns immediately.

### Monitoring progress

In [14]:
import asyncio

# Wait for the first DataEvent to confirm the method actually started
progress = await odtc.tc.wait_for_first_progress(timeout=60.0)
print("Protocol started:", progress)

CancelledError: 

In [15]:
# Poll progress every 5 seconds
for _ in range(6):
    progress = await odtc.tc.request_progress()
    if progress is not None:
        print(progress)
    await asyncio.sleep(5)

CancelledError: 

### Stopping a protocol

In [16]:
await odtc.tc.stop_protocol()

## Custom ramp rates

Control how fast the block transitions between steps using `Ramp(rate=...)`. The rate is in °C/s.

- `Ramp()` or `FULL_SPEED` — as fast as the device can go (hardware default)
- `Ramp(rate=4.4)` — maximum heating rate for 96-well variant
- `Ramp(rate=2.2)` — maximum cooling rate for 96-well variant
- `Ramp(rate=1.0)` — gentle ramp for sensitive applications

In [ ]:
precise_protocol = Protocol(
    name="PreciseRamps",
    stages=[
        Stage(
            steps=[
                Step(temperature=95.0, hold_seconds=30, ramp=Ramp(rate=4.4)),  # max heating
                Step(temperature=55.0, hold_seconds=30, ramp=Ramp(rate=2.2)),  # max cooling
                Step(temperature=72.0, hold_seconds=60, ramp=Ramp(rate=1.5)),  # gentle
            ],
            repeats=10,
        ),
    ],
)

await odtc.tc.run_protocol(
    precise_protocol,
    backend_params=ODTCThermocyclerBackend.RunProtocolParams(fluid_quantity=1),
)

## Per-step lid temperature

The heated lid temperature can be set globally on `Protocol.lid_temperature`, or overridden per step via `Step.lid_temperature`.

> **Note:** `lid_temperature` refers to the *heated cover* that prevents condensation — it is separate from the motorized `door` capability.

In [ ]:
lid_protocol = Protocol(
    name="PerStepLid",
    lid_temperature=105.0,   # default for all steps
    stages=[
        Stage(
            steps=[
                Step(temperature=95.0, hold_seconds=30),                         # lid = 105 °C (default)
                Step(temperature=55.0, hold_seconds=30, lid_temperature=100.0),  # override
                Step(temperature=72.0, hold_seconds=60, lid_temperature=110.0),  # override
            ],
            repeats=5,
        ),
    ],
)

## Overshoot control

The ODTC uses *overshoot* to achieve fast, precise ramp rates: the block briefly exceeds the target temperature, then settles to the exact plateau. The backend computes optimal overshoot parameters automatically.

You can also specify an explicit overshoot:

```python
Ramp(
    rate=4.4,
    overshoot=Overshoot(
        target_temp=5.5,    # °C above plateau to briefly reach
        hold_seconds=0.0,   # time at peak (0 = triangular pulse)
        return_rate=2.2,    # °C/s falling back to plateau
    )
)
```

When `overshoot` is `None` (the default), the backend computes it from temperature delta, ramp rate, and fluid quantity. To **disable** overshoot entirely, pass `apply_overshoot=False` to `RunProtocolParams` or `ODTCProtocol.from_protocol()`.

In [ ]:
# Auto-computed overshoot (recommended)
auto_overshoot_protocol = Protocol(
    stages=[
        Stage(
            steps=[
                Step(temperature=95.0, hold_seconds=30, ramp=Ramp(rate=4.4)),
                Step(temperature=55.0, hold_seconds=30, ramp=Ramp(rate=2.2)),
            ],
            repeats=35,
        ),
    ],
)

# Explicit overshoot override
explicit_overshoot_protocol = Protocol(
    stages=[
        Stage(
            steps=[
                Step(
                    temperature=95.0,
                    hold_seconds=30,
                    ramp=Ramp(
                        rate=4.4,
                        overshoot=Overshoot(
                            target_temp=6.0,
                            hold_seconds=0.0,
                            return_rate=2.2,
                        ),
                    ),
                ),
            ],
            repeats=1,
        )
    ],
)

## Nested cycling loops

`Stage.inner_stages` supports nested PCR loops. The ODTC encodes these natively using goto/loop firmware instructions.

Example: outer 30-cycle loop containing an inner 5-cycle loop:

In [ ]:
nested_protocol = Protocol(
    name="NestedCycles",
    stages=[
        Stage(
            steps=[Step(temperature=95.0, hold_seconds=10)],
            repeats=30,
            inner_stages=[
                Stage(
                    steps=[
                        Step(temperature=55.0, hold_seconds=10),
                        Step(temperature=72.0, hold_seconds=20),
                    ],
                    repeats=5,
                ),
            ],
        ),
        Stage(
            steps=[Step(temperature=50.0, hold_seconds=20)],
            repeats=30,
        ),
    ],
)

## Storing named protocols

You can compile and upload a protocol to the device once, then run it later by name — without recompiling or re-uploading. The method persists on the device across sessions until explicitly deleted.

Use `ODTCProtocol.from_protocol()` to compile the protocol with a name, then `upload_protocol()` to store it, and `run_stored_protocol()` to execute it later.

In [ ]:
# Compile the protocol to the ODTC's device-native form with a persistent name
odtc_protocol = ODTCProtocol.from_protocol(
    pcr_protocol,
    variant=96,
    fluid_quantity=FluidQuantity.UL_30_TO_74,
    name="StandardPCR",          # is_scratch=False: persists on device
    apply_overshoot=True,        # default; set False to disable auto-overshoot
)
print(odtc_protocol)

# Upload once — survives device Reset and can be run in future sessions
await odtc.tc.backend.upload_protocol(odtc_protocol)

In [ ]:
# List methods currently stored on the device
method_set = await odtc.tc.backend.get_method_set()
print(method_set)

# Retrieve a stored protocol by name (returns None if not found or is a premethod)
stored = await odtc.tc.backend.get_protocol("StandardPCR")
print(stored)

# Run a stored named protocol without re-uploading
# (works even after device Reset or reconnect)
await odtc.tc.backend.run_stored_protocol("StandardPCR")

## Variant: 384-well

The 384-well ODTC uses a higher maximum heating slope (5.0 °C/s vs 4.4 °C/s) and supports plate type 2.

In [ ]:
# odtc_384 = ODTC(odtc_ip="169.254.x.x", variant=384, name="odtc_384")
# await odtc_384.setup()
#
# await odtc_384.tc.run_protocol(
#     pcr_protocol,
#     backend_params=ODTCThermocyclerBackend.RunProtocolParams(
#         variant=384,
#         fluid_quantity=2,   # 75–100 µL
#         plate_type=0,
#     ),
# )

## Teardown

In [ ]:
await odtc.stop()  # deactivates block, closes SiLA connection